# **Building a model to classify breeds of dogs - I will use transfer learning to adapt *resnet18* model to this problem**

- **Dataset**: Stanford Dogs Dataset
- **Context**: The Stanford Dogs dataset contains images of 120 breeds of dogs from around the world. This dataset has been built using images and annotation from ImageNet for the task of fine-grained image categorization. It was originally collected for fine-grain image categorization, a challenging problem as certain dog breeds have near identical features or differ in colour and age.
- **Content**:
    - Number of categories: 120
    - Number of images: 20,580
    - Annotations: Class labels, Bounding boxes


# **1. Data**

In [18]:
!pip install torchmetrics

In [19]:
import torch
from torch import nn

from torch.utils.data import DataLoader, Subset
from torchvision import models, transforms, datasets
from torchmetrics.classification import Accuracy

import copy

from google.colab import drive

from sklearn.model_selection import train_test_split

import os

from pathlib import Path

In [20]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [22]:
data_dir = '/content/drive/MyDrive/data/stanford_dogs'

full_dataset_train = datasets.ImageFolder(root=data_dir, transform=train_transform)
full_dataset_val = datasets.ImageFolder(root=data_dir, transform=val_transform)

targets = full_dataset_train.targets
indices = list(range(len(targets)))

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2,
    stratify=targets,
    random_state=42
)

train_dataset = Subset(full_dataset_train, train_indices)
val_dataset = Subset(full_dataset_val, val_indices)

train_dataloader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True, num_workers=os.cpu_count())
val_dataloader = DataLoader(dataset=val_dataset, batch_size=32, shuffle=False, num_workers=os.cpu_count())

In [23]:
device = "cuda" if torch.cuda.is_available() else "cpu"


# **2. Model**

In [24]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(in_features=num_features, out_features=120)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
accuracy = Accuracy(task="multiclass", num_classes=120).to(device)

def train_model(model, criterion, optimizer, num_epochs=5):
    dataloaders = {
        'train': train_dataloader,
        'val': val_dataloader
    }

    datasets_sizes = {
        'train': len(train_dataloader.dataset),
        'val': len(val_dataloader.dataset)
    }
    best_weights = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f"Epoch: {epoch+1}/{num_epochs}")

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            accuracy.reset()

            for X_batch, y_batch in dataloaders[phase]:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase=='train'):
                    y_preds = model(X_batch)
                    loss = criterion(y_preds, y_batch)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * X_batch.size(0)
                accuracy.update(y_preds, y_batch)

            epoch_loss = running_loss / datasets_sizes[phase]
            epoch_acc = accuracy.compute().item()

            print(f'{phase} loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_weights = copy.deepcopy(model.state_dict())

        print('\n')

    print(f'Best val accuracy: {best_acc:.2f}')
    model.load_state_dict(best_weights)
    return model


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 198MB/s]


In [25]:
model = train_model(model=model, criterion=criterion, optimizer=optimizer, num_epochs=5)

Epoch: 1/5


KeyboardInterrupt: 

In [ ]:
for name, child in model.named_children():
    if name in ['layer3', 'layer4']:
        for param in child.parameters():
            param.requires_grad = True

optimizer_fine = torch.optim.Adam([
    {'params': model.layer3.parameters(), 'lr': 1e-5},
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(), 'lr': 1e-4}

])

model = train_model(model=model, criterion=criterion, optimizer=optimizer_fine, num_epochs=10)

In [ ]:
model_folder = Path('models')
model_folder.mkdir(parents=True, exist_ok=True)

model_name = 'stanford_dogs_model.pth'
torch.save(obj=model.state_dict(), f=model_folder/model_name)